# 02M — Isoforms & distal regions


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


Literary operationalization: `dp140_preserved` = exon<=44, `dp140_indeterminate` = exon 45-50, `dp140_affected` = exon>=51; `dp71_affected` = exon>=63; `distal` = exon>=45.


## 1. 📊 dp140 vs phenotype ⭐


In [3]:
tmp = d[['dp140_status', 'phenotype']].dropna()
fig = px.histogram(tmp, x='dp140_status', color='phenotype', barmode='stack', title='dp140 status vs phenotype')
fig.show()


## 2. 🧪 Fisher ⭐


In [4]:
tmp = d[d['dp140_status'].isin(['dp140_preserved', 'dp140_affected']) & d['phenotype'].isin(['DMD', 'BMD'])][['is_dp140_affected', 'phenotype']].dropna().copy()
tmp['is_dmd'] = tmp['phenotype'].eq('DMD')
tab, odds, p, ci = fisher_bool(tmp, 'is_dp140_affected', 'is_dmd')
display(tab)
display(pd.Series({'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]}).to_frame('value'))


is_dmd,False,True
is_dp140_affected,,
False,598,4993
True,338,2221


,value
odds_ratio,0.786994
p_value,0.001120
or_ci_low,0.682513
or_ci_high,0.907469


## 3. 📊 dp71 vs phenotype ⭐


In [5]:
tmp = d[['is_dp71_affected', 'phenotype']].dropna()
fig = px.histogram(tmp, x='is_dp71_affected', color='phenotype', barmode='stack', title='dp71 status vs phenotype')
fig.show()


## 4. 🧪 Fisher


In [6]:
tmp = d[d['phenotype'].isin(['DMD', 'BMD'])][['is_dp71_affected', 'phenotype']].dropna().copy()
tmp['is_dmd'] = tmp['phenotype'].eq('DMD')
tab, odds, p, ci = fisher_bool(tmp, 'is_dp71_affected', 'is_dmd')
display(tab)
display(pd.Series({'odds_ratio': odds, 'p_value': p, 'or_ci_low': ci[1], 'or_ci_high': ci[2]}).to_frame('value'))


is_dmd,False,True
is_dp71_affected,,
False,845,6857
True,178,950


,value
odds_ratio,0.657697
p_value,0.000006
or_ci_low,0.551940
or_ci_high,0.783720


## 5. 📊 dp140 vs pathogenic


In [7]:
tmp = d[['dp140_status', 'pathogenic']].dropna()
fig = px.histogram(tmp, x='dp140_status', color='pathogenic', barmode='stack', title='dp140 status vs pathogenic')
fig.show()


## 6. 📊 dp71 vs pathogenic


In [8]:
tmp = d[['is_dp71_affected', 'pathogenic']].dropna()
fig = px.histogram(tmp, x='is_dp71_affected', color='pathogenic', barmode='stack', title='dp71 status vs pathogenic')
fig.show()


## 7. 📊 distal vs mutation_type ⭐


In [9]:
tmp = d[['is_distal_region', 'mutation_type']].dropna()
fig = px.histogram(tmp, x='is_distal_region', color='mutation_type', barmode='stack', title='Distal region vs mutation_type')
fig.show()


## 8. 🧪 χ² ⭐


In [10]:
tab, chi2, p, dof = chi2_table(d, 'is_distal_region', 'mutation_type')
display(tab)
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


mutation_type,frameshift,large_deletion,missense,nonsense,splice
is_distal_region,,,,,
False,481,962,2256,553,562
True,273,739,1072,238,378


,value
chi2,8.095679e+01
dof,4.000000e+00
p_value,1.092141e-16


## 9. 📊 distal vs frame


In [11]:
tmp = d[['is_distal_region', 'frame']].dropna()
fig = px.histogram(tmp, x='is_distal_region', color='frame', barmode='stack', title='Distal region vs frame')
fig.show()


## 10. 🧪 χ²


In [12]:
tab, chi2, p, dof = chi2_table(d, 'is_distal_region', 'frame')
display(tab)
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


frame,in-frame,out-of-frame
is_distal_region,,
False,2256,2558
True,1072,1628


,value
chi2,3.564705e+01
dof,1.000000e+00
p_value,2.365052e-09
